In [1]:
import pandas as pd
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/My Drive/CADE

/content/drive/My Drive/CADE


## Main results

### Forecasting

In [4]:
"""
Evaluation Script: Global Length-Matched Subset MSE & MAE
=========================================================
1. Parse all three model CSVs
2. Find rows where ALL models produce output length == ground-truth length
3. Compute MSE and MAE per model on that shared subset
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"")
        try:
            result.append(float(item))
        except ValueError:
            result.append(None)
    return [x for x in result if x is not None]


def extract_single_number(text):
    nums = re.findall(r'[-+]?\d*\.?\d+', text)
    if nums:
        return [float(nums[-1])]
    return None


def parse_answer(qa_text):
    if '"answer"' not in qa_text:
        return None
    answer_part = qa_text.split('"answer"')[-1]
    answer_list = extract_list_from_text(answer_part)
    if answer_list and len(answer_list) > 0:
        return answer_list
    ans_match = re.search(r'"answer".*?:\s*"(.*?)"', qa_text, re.DOTALL)
    if ans_match:
        return extract_single_number(ans_match.group(1))
    return None


def parse_model_response(model_text):
    model_list = extract_list_from_text(model_text)
    if model_list and len(model_list) > 0:
        return model_list
    return extract_single_number(model_text)


def parse_csv(csv_path):
    """Parse a CSV and return list of (answer_list, model_list) per row."""
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column
            answer_list = parse_answer(qa_text)
            model_list = parse_model_response(model_text)
            rows.append((answer_list, model_list))
    return rows


def find_global_matched_indices(all_model_rows):
    """
    Find row indices where ALL models have:
      - successfully parsed answer and prediction
      - prediction length == answer length
    """
    n_rows = min(len(rows) for rows in all_model_rows)
    valid_indices = []

    for i in range(n_rows):
        all_valid = True
        for rows in all_model_rows:
            ans, pred = rows[i]
            if ans is None or pred is None:
                all_valid = False
                break
            if len(ans) == 0 or len(pred) == 0:
                all_valid = False
                break
            if len(ans) != len(pred):
                all_valid = False
                break
        if all_valid:
            valid_indices.append(i)

    return valid_indices


def compute_metrics(rows, indices):
    """Compute MSE and MAE on the given subset of row indices."""
    all_se = []
    all_ae = []
    row_mses = []
    row_maes = []

    for i in indices:
        ans, pred = rows[i]
        n = len(ans)  # guaranteed equal length in valid subset
        se = [(ans[j] - pred[j]) ** 2 for j in range(n)]
        ae = [abs(ans[j] - pred[j]) for j in range(n)]
        all_se.extend(se)
        all_ae.extend(ae)
        row_mses.append(np.mean(se))
        row_maes.append(np.mean(ae))

    return {
        "mse_all_pairs": float(np.mean(all_se)) if all_se else None,
        "mae_all_pairs": float(np.mean(all_ae)) if all_ae else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "mae_avg_per_row": float(np.mean(row_maes)) if row_maes else None,
        "total_value_pairs": len(all_se),
        "rows_used": len(indices),
    }


if __name__ == "__main__":
    model_files = {
        "Time-MoE":      "/content/drive/My Drive/CADE/results/time_moe_6tasks/results_forecasting.csv",
        "Frozen Random Linear":      "/content/drive/My Drive/CADE/results/random_frozen_6tasks/results_forecasting.csv",
        "Time-MQA":    "/content/drive/My Drive/CADE/results/qwen3-0.6B+lora/results_forecasting.csv",
        "Time-MQA(Full FT)":    "/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_forecasting.csv",
        "CADE w/o SupCon":    "/content/drive/My Drive/CADE/results/emb_lr0_random_trainable_6tasks/results_forecasting.csv",
        "CADE":    "/content/drive/My Drive/CADE/results/qsize_512_w0.1/results_forecasting.csv",
    }

    # Step 1: Parse all CSVs
    all_parsed = {}
    for name, path in model_files.items():
        all_parsed[name] = parse_csv(path)
        print(f"Parsed {name}: {len(all_parsed[name])} rows")

    # Step 2: Find global length-matched subset
    valid_indices = find_global_matched_indices(list(all_parsed.values()))
    total_rows = min(len(v) for v in all_parsed.values())
    print(f"\nGlobal valid subset: {len(valid_indices)} / {total_rows} rows "
          f"({100*len(valid_indices)/total_rows:.1f}%)\n")

    # Step 3: Evaluate each model on the shared subset
    print("=" * 60)
    print("  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET")
    print("=" * 60)
    print(f"  Shared valid rows: {len(valid_indices)}")
    print("-" * 60)
    print(f"  {'Model':<15} {'MSE (all pairs)':>18} {'MAE (all pairs)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")

    results_summary = {}
    for name in model_files:
        metrics = compute_metrics(all_parsed[name], valid_indices)
        results_summary[name] = metrics
        print(f"  {name:<15} {metrics['mse_all_pairs']:>18.6f} {metrics['mae_all_pairs']:>18.6f}")

    print("-" * 60)
    print(f"\n  {'Model':<15} {'MSE (avg/row)':>18} {'MAE (avg/row)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")
    for name in model_files:
        m = results_summary[name]
        print(f"  {name:<15} {m['mse_avg_per_row']:>18.6f} {m['mae_avg_per_row']:>18.6f}")

    print("=" * 60)

    # Also show per-model individual parse stats for context
    print("\n  Per-model parse diagnostics (before global filtering):")
    for name in model_files:
        rows = all_parsed[name]
        parsed_ok = sum(1 for a, p in rows if a is not None and p is not None)
        len_match = sum(1 for a, p in rows if a is not None and p is not None and len(a) == len(p))
        print(f"    {name:<15}  parseable: {parsed_ok}/{len(rows)},  length-matched: {len_match}/{len(rows)}")

Parsed Time-MoE: 379 rows
Parsed Frozen Random Linear: 379 rows
Parsed Time-MQA: 379 rows
Parsed Time-MQA(Full FT): 379 rows
Parsed CADE w/o SupCon: 379 rows
Parsed CADE: 379 rows

Global valid subset: 79 / 379 rows (20.8%)

  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET
  Shared valid rows: 79
------------------------------------------------------------
  Model              MSE (all pairs)    MAE (all pairs)
  ............... .................. ..................
  Time-MoE              41100.090056          52.728735
  Frozen Random Linear       30674.096263          48.552163
  Time-MQA              30965.328496          48.153932
  Time-MQA(Full FT)     3699367.287995         277.443941
  CADE w/o SupCon       38147.098469          52.791366
  CADE                  33043.537395          49.184795
------------------------------------------------------------

  Model                MSE (avg/row)      MAE (avg/row)
  ............... .................. ..................
  Tim

### Imputation

In [5]:
"""
Evaluation Script: Global Length-Matched Subset MSE & MAE
=========================================================
1. Parse all three model CSVs
2. Find rows where ALL models produce output length == ground-truth length
3. Compute MSE and MAE per model on that shared subset
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"")
        try:
            result.append(float(item))
        except ValueError:
            result.append(None)
    return [x for x in result if x is not None]


def extract_single_number(text):
    nums = re.findall(r'[-+]?\d*\.?\d+', text)
    if nums:
        return [float(nums[-1])]
    return None


def parse_answer(qa_text):
    if '"answer"' not in qa_text:
        return None
    answer_part = qa_text.split('"answer"')[-1]
    answer_list = extract_list_from_text(answer_part)
    if answer_list and len(answer_list) > 0:
        return answer_list
    ans_match = re.search(r'"answer".*?:\s*"(.*?)"', qa_text, re.DOTALL)
    if ans_match:
        return extract_single_number(ans_match.group(1))
    return None


def parse_model_response(model_text):
    model_list = extract_list_from_text(model_text)
    if model_list and len(model_list) > 0:
        return model_list
    return extract_single_number(model_text)


def parse_csv(csv_path):
    """Parse a CSV and return list of (answer_list, model_list) per row."""
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column
            answer_list = parse_answer(qa_text)
            model_list = parse_model_response(model_text)
            rows.append((answer_list, model_list))
    return rows


def find_global_matched_indices(all_model_rows):
    """
    Find row indices where ALL models have:
      - successfully parsed answer and prediction
      - prediction length == answer length
    """
    n_rows = min(len(rows) for rows in all_model_rows)
    valid_indices = []

    for i in range(n_rows):
        all_valid = True
        for rows in all_model_rows:
            ans, pred = rows[i]
            if ans is None or pred is None:
                all_valid = False
                break
            if len(ans) == 0 or len(pred) == 0:
                all_valid = False
                break
            if len(ans) != len(pred):
                all_valid = False
                break
        if all_valid:
            valid_indices.append(i)

    return valid_indices


def compute_metrics(rows, indices):
    """Compute MSE and MAE on the given subset of row indices."""
    all_se = []
    all_ae = []
    row_mses = []
    row_maes = []

    for i in indices:
        ans, pred = rows[i]
        n = len(ans)  # guaranteed equal length in valid subset
        se = [(ans[j] - pred[j]) ** 2 for j in range(n)]
        ae = [abs(ans[j] - pred[j]) for j in range(n)]
        all_se.extend(se)
        all_ae.extend(ae)
        row_mses.append(np.mean(se))
        row_maes.append(np.mean(ae))

    return {
        "mse_all_pairs": float(np.mean(all_se)) if all_se else None,
        "mae_all_pairs": float(np.mean(all_ae)) if all_ae else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "mae_avg_per_row": float(np.mean(row_maes)) if row_maes else None,
        "total_value_pairs": len(all_se),
        "rows_used": len(indices),
    }


if __name__ == "__main__":
    model_files = {
        "Time-MoE":      "/content/drive/My Drive/CADE/results/time_moe_6tasks/results_imputation.csv",
        "Frozen Random Linear":      "/content/drive/My Drive/CADE/results/random_frozen_6tasks/results_imputation.csv",
        "Time-MQA":    "/content/drive/My Drive/CADE/results/qwen3-0.6B+lora/results_imputation.csv",
        "Time-MQA(Full FT)":    "/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_imputation.csv",
        "CADE w/o SupCon":    "/content/drive/My Drive/CADE/results/emb_lr0_random_trainable_6tasks/results_imputation.csv",
        "CADE":    "/content/drive/My Drive/CADE/results/qsize_512_w0.1/results_imputation.csv",
    }

    # Step 1: Parse all CSVs
    all_parsed = {}
    for name, path in model_files.items():
        all_parsed[name] = parse_csv(path)
        print(f"Parsed {name}: {len(all_parsed[name])} rows")

    # Step 2: Find global length-matched subset
    valid_indices = find_global_matched_indices(list(all_parsed.values()))
    total_rows = min(len(v) for v in all_parsed.values())
    print(f"\nGlobal valid subset: {len(valid_indices)} / {total_rows} rows "
          f"({100*len(valid_indices)/total_rows:.1f}%)\n")

    # Step 3: Evaluate each model on the shared subset
    print("=" * 60)
    print("  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET")
    print("=" * 60)
    print(f"  Shared valid rows: {len(valid_indices)}")
    print("-" * 60)
    print(f"  {'Model':<15} {'MSE (all pairs)':>18} {'MAE (all pairs)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")

    results_summary = {}
    for name in model_files:
        metrics = compute_metrics(all_parsed[name], valid_indices)
        results_summary[name] = metrics
        print(f"  {name:<15} {metrics['mse_all_pairs']:>18.6f} {metrics['mae_all_pairs']:>18.6f}")

    print("-" * 60)
    print(f"\n  {'Model':<15} {'MSE (avg/row)':>18} {'MAE (avg/row)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")
    for name in model_files:
        m = results_summary[name]
        print(f"  {name:<15} {m['mse_avg_per_row']:>18.6f} {m['mae_avg_per_row']:>18.6f}")

    print("=" * 60)

    # Also show per-model individual parse stats for context
    print("\n  Per-model parse diagnostics (before global filtering):")
    for name in model_files:
        rows = all_parsed[name]
        parsed_ok = sum(1 for a, p in rows if a is not None and p is not None)
        len_match = sum(1 for a, p in rows if a is not None and p is not None and len(a) == len(p))
        print(f"    {name:<15}  parseable: {parsed_ok}/{len(rows)},  length-matched: {len_match}/{len(rows)}")

Parsed Time-MoE: 400 rows
Parsed Frozen Random Linear: 400 rows
Parsed Time-MQA: 400 rows
Parsed Time-MQA(Full FT): 400 rows
Parsed CADE w/o SupCon: 400 rows
Parsed CADE: 400 rows

Global valid subset: 148 / 400 rows (37.0%)

  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET
  Shared valid rows: 148
------------------------------------------------------------
  Model              MSE (all pairs)    MAE (all pairs)
  ............... .................. ..................
  Time-MoE               5501.476188           2.859733
  Frozen Random Linear        5403.994977           2.797450
  Time-MQA               4779.095250           2.525342
  Time-MQA(Full FT)        4016.824230           2.456692
  CADE w/o SupCon        5687.893387           2.829272
  CADE                   5629.041954           2.900944
------------------------------------------------------------

  Model                MSE (avg/row)      MAE (avg/row)
  ............... .................. ..................
  T

## With Deepseek

### Forecasting

In [4]:
"""
Evaluation Script: Global Length-Matched Subset MSE & MAE
=========================================================
1. Parse all three model CSVs
2. Find rows where ALL models produce output length == ground-truth length
3. Compute MSE and MAE per model on that shared subset
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"")
        try:
            result.append(float(item))
        except ValueError:
            result.append(None)
    return [x for x in result if x is not None]


def extract_single_number(text):
    nums = re.findall(r'[-+]?\d*\.?\d+', text)
    if nums:
        return [float(nums[-1])]
    return None


def parse_answer(qa_text):
    if '"answer"' not in qa_text:
        return None
    answer_part = qa_text.split('"answer"')[-1]
    answer_list = extract_list_from_text(answer_part)
    if answer_list and len(answer_list) > 0:
        return answer_list
    ans_match = re.search(r'"answer".*?:\s*"(.*?)"', qa_text, re.DOTALL)
    if ans_match:
        return extract_single_number(ans_match.group(1))
    return None


def parse_model_response(model_text):
    model_list = extract_list_from_text(model_text)
    if model_list and len(model_list) > 0:
        return model_list
    return extract_single_number(model_text)


def parse_csv(csv_path):
    """Parse a CSV and return list of (answer_list, model_list) per row."""
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column
            answer_list = parse_answer(qa_text)
            model_list = parse_model_response(model_text)
            rows.append((answer_list, model_list))
    return rows


def find_global_matched_indices(all_model_rows):
    """
    Find row indices where ALL models have:
      - successfully parsed answer and prediction
      - prediction length == answer length
    """
    n_rows = min(len(rows) for rows in all_model_rows)
    valid_indices = []

    for i in range(n_rows):
        all_valid = True
        for rows in all_model_rows:
            ans, pred = rows[i]
            if ans is None or pred is None:
                all_valid = False
                break
            if len(ans) == 0 or len(pred) == 0:
                all_valid = False
                break
            if len(ans) != len(pred):
                all_valid = False
                break
        if all_valid:
            valid_indices.append(i)

    return valid_indices


def compute_metrics(rows, indices):
    """Compute MSE and MAE on the given subset of row indices."""
    all_se = []
    all_ae = []
    row_mses = []
    row_maes = []

    for i in indices:
        ans, pred = rows[i]
        n = len(ans)  # guaranteed equal length in valid subset
        se = [(ans[j] - pred[j]) ** 2 for j in range(n)]
        ae = [abs(ans[j] - pred[j]) for j in range(n)]
        all_se.extend(se)
        all_ae.extend(ae)
        row_mses.append(np.mean(se))
        row_maes.append(np.mean(ae))

    return {
        "mse_all_pairs": float(np.mean(all_se)) if all_se else None,
        "mae_all_pairs": float(np.mean(all_ae)) if all_ae else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "mae_avg_per_row": float(np.mean(row_maes)) if row_maes else None,
        "total_value_pairs": len(all_se),
        "rows_used": len(indices),
    }


if __name__ == "__main__":
    model_files = {
        "deepseek V3.2":      "/content/drive/My Drive/CADE/results/deepseekv3.2/results_forecasting.csv",
        "CADE":    "/content/drive/My Drive/CADE/results/qsize_512_w0.1/results_forecasting.csv",
    }

    # Step 1: Parse all CSVs
    all_parsed = {}
    for name, path in model_files.items():
        all_parsed[name] = parse_csv(path)
        print(f"Parsed {name}: {len(all_parsed[name])} rows")

    # Step 2: Find global length-matched subset
    valid_indices = find_global_matched_indices(list(all_parsed.values()))
    total_rows = min(len(v) for v in all_parsed.values())
    print(f"\nGlobal valid subset: {len(valid_indices)} / {total_rows} rows "
          f"({100*len(valid_indices)/total_rows:.1f}%)\n")

    # Step 3: Evaluate each model on the shared subset
    print("=" * 60)
    print("  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET")
    print("=" * 60)
    print(f"  Shared valid rows: {len(valid_indices)}")
    print("-" * 60)
    print(f"  {'Model':<15} {'MSE (all pairs)':>18} {'MAE (all pairs)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")

    results_summary = {}
    for name in model_files:
        metrics = compute_metrics(all_parsed[name], valid_indices)
        results_summary[name] = metrics
        print(f"  {name:<15} {metrics['mse_all_pairs']:>18.6f} {metrics['mae_all_pairs']:>18.6f}")

    print("-" * 60)
    print(f"\n  {'Model':<15} {'MSE (avg/row)':>18} {'MAE (avg/row)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")
    for name in model_files:
        m = results_summary[name]
        print(f"  {name:<15} {m['mse_avg_per_row']:>18.6f} {m['mae_avg_per_row']:>18.6f}")

    print("=" * 60)

    # Also show per-model individual parse stats for context
    print("\n  Per-model parse diagnostics (before global filtering):")
    for name in model_files:
        rows = all_parsed[name]
        parsed_ok = sum(1 for a, p in rows if a is not None and p is not None)
        len_match = sum(1 for a, p in rows if a is not None and p is not None and len(a) == len(p))
        print(f"    {name:<15}  parseable: {parsed_ok}/{len(rows)},  length-matched: {len_match}/{len(rows)}")

Parsed deepseek V3.2: 379 rows
Parsed CADE: 379 rows

Global valid subset: 221 / 379 rows (58.3%)

  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET
  Shared valid rows: 221
------------------------------------------------------------
  Model              MSE (all pairs)    MAE (all pairs)
  ............... .................. ..................
  deepseek V3.2         96835.485733          63.914851
  CADE                  74969.557341          50.368312
------------------------------------------------------------

  Model                MSE (avg/row)      MAE (avg/row)
  ............... .................. ..................
  deepseek V3.2         85688.497793          57.871917
  CADE                  82458.022607          50.309610

  Per-model parse diagnostics (before global filtering):
    deepseek V3.2    parseable: 379/379,  length-matched: 373/379
    CADE             parseable: 379/379,  length-matched: 227/379


### Imputation

In [5]:
"""
Evaluation Script: Global Length-Matched Subset MSE & MAE
=========================================================
1. Parse all three model CSVs
2. Find rows where ALL models produce output length == ground-truth length
3. Compute MSE and MAE per model on that shared subset
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"")
        try:
            result.append(float(item))
        except ValueError:
            result.append(None)
    return [x for x in result if x is not None]


def extract_single_number(text):
    nums = re.findall(r'[-+]?\d*\.?\d+', text)
    if nums:
        return [float(nums[-1])]
    return None


def parse_answer(qa_text):
    if '"answer"' not in qa_text:
        return None
    answer_part = qa_text.split('"answer"')[-1]
    answer_list = extract_list_from_text(answer_part)
    if answer_list and len(answer_list) > 0:
        return answer_list
    ans_match = re.search(r'"answer".*?:\s*"(.*?)"', qa_text, re.DOTALL)
    if ans_match:
        return extract_single_number(ans_match.group(1))
    return None


def parse_model_response(model_text):
    model_list = extract_list_from_text(model_text)
    if model_list and len(model_list) > 0:
        return model_list
    return extract_single_number(model_text)


def parse_csv(csv_path):
    """Parse a CSV and return list of (answer_list, model_list) per row."""
    rows = []
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column
            answer_list = parse_answer(qa_text)
            model_list = parse_model_response(model_text)
            rows.append((answer_list, model_list))
    return rows


def find_global_matched_indices(all_model_rows):
    """
    Find row indices where ALL models have:
      - successfully parsed answer and prediction
      - prediction length == answer length
    """
    n_rows = min(len(rows) for rows in all_model_rows)
    valid_indices = []

    for i in range(n_rows):
        all_valid = True
        for rows in all_model_rows:
            ans, pred = rows[i]
            if ans is None or pred is None:
                all_valid = False
                break
            if len(ans) == 0 or len(pred) == 0:
                all_valid = False
                break
            if len(ans) != len(pred):
                all_valid = False
                break
        if all_valid:
            valid_indices.append(i)

    return valid_indices


def compute_metrics(rows, indices):
    """Compute MSE and MAE on the given subset of row indices."""
    all_se = []
    all_ae = []
    row_mses = []
    row_maes = []

    for i in indices:
        ans, pred = rows[i]
        n = len(ans)  # guaranteed equal length in valid subset
        se = [(ans[j] - pred[j]) ** 2 for j in range(n)]
        ae = [abs(ans[j] - pred[j]) for j in range(n)]
        all_se.extend(se)
        all_ae.extend(ae)
        row_mses.append(np.mean(se))
        row_maes.append(np.mean(ae))

    return {
        "mse_all_pairs": float(np.mean(all_se)) if all_se else None,
        "mae_all_pairs": float(np.mean(all_ae)) if all_ae else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "mae_avg_per_row": float(np.mean(row_maes)) if row_maes else None,
        "total_value_pairs": len(all_se),
        "rows_used": len(indices),
    }


if __name__ == "__main__":
    model_files = {
        "deepseek V3.2":      "/content/drive/My Drive/CADE/results/deepseekv3.2/results_imputation.csv",
        "CADE":    "/content/drive/My Drive/CADE/results/qsize_512_w0.1/results_imputation.csv",
    }

    # Step 1: Parse all CSVs
    all_parsed = {}
    for name, path in model_files.items():
        all_parsed[name] = parse_csv(path)
        print(f"Parsed {name}: {len(all_parsed[name])} rows")

    # Step 2: Find global length-matched subset
    valid_indices = find_global_matched_indices(list(all_parsed.values()))
    total_rows = min(len(v) for v in all_parsed.values())
    print(f"\nGlobal valid subset: {len(valid_indices)} / {total_rows} rows "
          f"({100*len(valid_indices)/total_rows:.1f}%)\n")

    # Step 3: Evaluate each model on the shared subset
    print("=" * 60)
    print("  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET")
    print("=" * 60)
    print(f"  Shared valid rows: {len(valid_indices)}")
    print("-" * 60)
    print(f"  {'Model':<15} {'MSE (all pairs)':>18} {'MAE (all pairs)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")

    results_summary = {}
    for name in model_files:
        metrics = compute_metrics(all_parsed[name], valid_indices)
        results_summary[name] = metrics
        print(f"  {name:<15} {metrics['mse_all_pairs']:>18.6f} {metrics['mae_all_pairs']:>18.6f}")

    print("-" * 60)
    print(f"\n  {'Model':<15} {'MSE (avg/row)':>18} {'MAE (avg/row)':>18}")
    print(f"  {'':.<15} {'':.<18} {'':.<18}")
    for name in model_files:
        m = results_summary[name]
        print(f"  {name:<15} {m['mse_avg_per_row']:>18.6f} {m['mae_avg_per_row']:>18.6f}")

    print("=" * 60)

    # Also show per-model individual parse stats for context
    print("\n  Per-model parse diagnostics (before global filtering):")
    for name in model_files:
        rows = all_parsed[name]
        parsed_ok = sum(1 for a, p in rows if a is not None and p is not None)
        len_match = sum(1 for a, p in rows if a is not None and p is not None and len(a) == len(p))
        print(f"    {name:<15}  parseable: {parsed_ok}/{len(rows)},  length-matched: {len_match}/{len(rows)}")

Parsed deepseek V3.2: 400 rows
Parsed CADE: 400 rows

Global valid subset: 288 / 400 rows (72.0%)

  FORECASTING EVALUATION — GLOBAL LENGTH-MATCHED SUBSET
  Shared valid rows: 288
------------------------------------------------------------
  Model              MSE (all pairs)    MAE (all pairs)
  ............... .................. ..................
  deepseek V3.2         81579.952330           4.184801
  CADE                  24595.455592           3.588229
------------------------------------------------------------

  Model                MSE (avg/row)      MAE (avg/row)
  ............... .................. ..................
  deepseek V3.2        120663.183950           5.346907
  CADE                  31200.507374           4.319481

  Per-model parse diagnostics (before global filtering):
    deepseek V3.2    parseable: 400/400,  length-matched: 364/400
    CADE             parseable: 400/400,  length-matched: 314/400
